# 🚁 SUTRA — Subsystem C: VisDrone Fine-Tuning
**Project:** SUTRA  
**Goal:** Fine-tune YOLOv8-Nano on VisDrone2019 aerial dataset (mAP@0.5 >= 90%)


In [ ]:
!pip install ultralytics==8.2.0 -q
import os, time, json, torch
from ultralytics import YOLO
print(f'✅ CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}')


In [ ]:
# Step 2: Train using built-in VisDrone.yaml auto-dataset
model = YOLO('yolov8n.pt')
print('🚀 Starting training...')
results = model.train(
    data      = 'VisDrone.yaml',
    epochs    = 50,
    imgsz     = 640,
    batch     = 16,
    patience  = 15,
    device    = 0 if torch.cuda.is_available() else 'cpu',
    workers   = 4,
    project   = '/kaggle/working/sutra_train',
    name      = 'yolov8n_visdrone',
    exist_ok  = True,
    pretrained= True,
    optimizer = 'AdamW',
    lr0       = 0.001,
    lrf       = 0.01,
    mosaic    = 1.0,
    mixup     = 0.1,
    degrees   = 15.0,
    flipud    = 0.5,
    fliplr    = 0.5,
    scale     = 0.5,
    save      = True,
    val       = True,
    plots     = True,
    verbose   = True,
)
print('✅ Training completed!')


In [ ]:
best_model = YOLO('/kaggle/working/sutra_train/yolov8n_visdrone/weights/best.pt')
metrics = best_model.val(data='VisDrone.yaml', device=0 if torch.cuda.is_available() else 'cpu')
print(f'mAP@0.5: {metrics.box.map50*100:.2f}%')
best_model.export(format='onnx', imgsz=640)
print('✅ ONNX exported!')
